# 05 - End-to-End Evaluation
## Text-to-SQL SLM Research Architecture (Phase 4: Evaluation & Benchmarking)

Runs the real evaluation harness (`src/evaluation/evaluate_end_to_end.py`) on a trained generator checkpoint:
1. **Google Drive Integration**: mounts `/content/drive` and finds trained checkpoints.
2. **Candidate generation**: samples K rollouts from the checkpoint. The generator never sees gold SQL.
3. **Validation**: grammar check, PV-SQL 10 AST rules, DB probing, System-1 scorer
   (trained model if `SCORER_CHECKPOINT` is set, otherwise the keyword heuristic baseline, reported as `scorer_type`).
4. **Metrics**: EX (BIRD official set comparison, 30 s timeout), VSR, ECE, latency, peak VRAM.

Gold SQL is used only to score predictions. If no checkpoint is found the notebook stops; it never fabricates results.


In [ ]:
# Step 1: Workspace Path Setup, Environment Setup & Google Drive Mounting
import os
import sys

WORKSPACE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists("../src") else os.path.abspath(".")
if WORKSPACE_DIR not in sys.path:
    sys.path.insert(0, WORKSPACE_DIR)

import json
import time
import math
import sqlite3
import subprocess

try:
    import torch
    gpu_available = torch.cuda.is_available()
    gpu_device_name = torch.cuda.get_device_name(0) if gpu_available else None
except ImportError:
    torch = None
    gpu_available = False
    gpu_device_name = None

print("=== Colab Evaluation Environment Verification ===")
print(f"CUDA Available: {gpu_available}")
if gpu_device_name:
    print(f"GPU Device: {gpu_device_name}")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS_DIR = "/content/drive/MyDrive/Text2SQL_Colab/results"
    DRIVE_CHECKPOINTS_ROOT = "/content/drive/MyDrive/Text2SQL_Colab/checkpoints"
    print("Google Drive mounted at /content/drive")
except (ImportError, Exception):
    DRIVE_RESULTS_DIR = os.path.abspath("./colab_drive_mock/results")
    DRIVE_CHECKPOINTS_ROOT = os.path.abspath("./colab_drive_mock/checkpoints")
    print(f"Using local Drive path: {DRIVE_RESULTS_DIR}")

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results Output Path: {DRIVE_RESULTS_DIR}")

In [ ]:
# Step 2: Checkpoint Discovery & Session Recovery
from src.training.callbacks import find_latest_checkpoint

# Look for latest checkpoints across training runs
sft_checkpoint = find_latest_checkpoint(os.path.join(DRIVE_CHECKPOINTS_ROOT, "sft_jepa_3b"))
grpo_checkpoint = find_latest_checkpoint(os.path.join(DRIVE_CHECKPOINTS_ROOT, "grpo_3b"))

print(f"Latest SFT Checkpoint:  {sft_checkpoint}")
print(f"Latest GRPO Checkpoint: {grpo_checkpoint}")


In [ ]:
# Step 3: Run the end-to-end evaluation harness on the trained generator
from src.evaluation import EvaluationHarness, format_benchmark_summary

GENERATOR_CHECKPOINT = sft_checkpoint  # or grpo_checkpoint, or an explicit path
SCORER_CHECKPOINT = None               # path to a trained System-1 scorer, if any
DATA_BIRD = "data/mini/mini_bird.jsonl"  # replace with BIRD dev.json converted to JSONL
DB_DIR = "data/mini"                     # replace with the BIRD dev_databases folder

if not GENERATOR_CHECKPOINT:
    raise RuntimeError("No generator checkpoint found. Train with notebook 02 first.")

harness = EvaluationHarness(
    db_dir=DB_DIR,
    num_candidates=8,
    model_path=GENERATOR_CHECKPOINT,
    scorer_model_path=SCORER_CHECKPOINT,
    output_log_dir=DRIVE_RESULTS_DIR,
    output_report_dir=DRIVE_RESULTS_DIR,
)
results = harness.evaluate_dataset(DATA_BIRD)
print(format_benchmark_summary(results, title="End-to-End Evaluation"))


In [ ]:
# Step 4: Save the measured results with provenance
results["generator_checkpoint"] = GENERATOR_CHECKPOINT
results["scorer_checkpoint"] = SCORER_CHECKPOINT
results["dataset_path"] = DATA_BIRD
results["export_timestamp"] = time.strftime("%Y-%m-%dT%H:%M:%SZ")

report_path = os.path.join(DRIVE_RESULTS_DIR, "end_to_end_benchmark_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(f"Report exported: {report_path}")
